In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import yaml
import copy

from home_energy.load_esb import load_hdf
from home_energy.tariff import bill_summary, price_per_kwh

df = load_hdf("../data/raw/hdf_kwh_30min.csv")

with open("../config/system.yaml") as f:
    cfg = yaml.safe_load(f)

plans = cfg["tariffs"]
charges = cfg["regulated_charges"]
plan = plans["electric_ireland_current"]

In [ ]:
def with_discount(plan, pct):
    p = copy.deepcopy(plan)
    p["discounts"] = ([{"label": f"{pct}%", "pct": pct, "applies_to": "import_units",
                        "valid_to": datetime.date(2027, 12, 5)}] if pct else [])
    return p

year = df.loc["2025-11-01":"2026-10-31"]
stay = with_discount(plans["electric_ireland_current"], 5.5)   # The 24% ends 5 Dec and goes to 5.5

rows = []
for name, p in plans.items():
    p_used = stay if name == "electric_ireland_current" else p
    yr1 = bill_summary(p_used, year, days=365, charges=charges)
    ongoing = bill_summary(p, year, days=365, charges=charges, apply_discounts=False)
    rows.append({"plan": name, "year1": yr1["net_cost"], "ongoing": ongoing["net_cost"],
                 "import": yr1["import_cost"], "standing": yr1["standing"],
                 "export_credit": yr1["export_credit"]})
table = pd.DataFrame(rows).set_index("plan")

baseline = table.loc["electric_ireland_current", "year1"]
table["saving_year1"] = baseline - table["year1"]
table["saving_ongoing"] = baseline - table["ongoing"]
table = table.sort_values("saving_year1", ascending=False).round(0)
table

In [ ]:
ax = table[["year1", "ongoing"]].sort_values("ongoing").plot(kind="barh", figsize=(9, 5), title="Annual net cost by plan (euros)")
ax.axvline(baseline, color="grey", linestyle="--")

In [ ]:
def scenario(usage, export_mult=1.0, import_mult=1.0):
    u = usage.copy()
    u["export_kwh"] *= export_mult
    u["import_kwh"] *= import_mult
    return u

def add_night_load(usage, kwh_per_year, hours=(2, 3, 4)):
    u = usage.copy()
    mask = u.index.hour.isin(hours)
    u.loc[mask, "import_kwh"] += kwh_per_year / mask.sum()
    return u

def ongoing_costs(u):
    return {name: bill_summary(p, u, days=365, charges=charges, apply_discounts=False)["net_cost"]
            for name, p in plans.items()}

scenarios = {
    "base": year,
    "export -20%": scenario(year, export_mult=0.8),
    "export +20%": scenario(year, export_mult=1.2),
    "usage -20%": scenario(year, import_mult=0.8),
    "usage +20%": scenario(year, import_mult=1.2)
}
costs = pd.DataFrame({label: ongoing_costs(u) for label, u in scenarios.items()}).round(0)
costs.rank().astype(int)                                         
costs.sub(costs.loc["electric_ireland_current"], axis=1)

In [ ]:
def net(plan, pct):
    return bill_summary(with_discount(plan, pct), year, days=365, charges=charges)["net_cost"]

def breakeven_pct(plan, target):
    c0, c10 = net(plan, 0), net(plan, 10)
    return (c0 - target) / ((c0 - c10) / 10)

sst = plans["electric_ireland_current"]
nb = plans["electric_ireland_night_boost"]
rivals = [r for r in table.index if not r.startswith("electric_ireland_current")]

cutoff = pd.DataFrame({
    "rival_year1": table.loc[rivals, "year1"],
    "SST % to match": [breakeven_pct(sst, table.loc[r, "year1"]) for r in rivals],
    "Night Boost % to match": [breakeven_pct(nb, table.loc[r, "year1"]) for r in rivals],
}).round(1).sort_values("rival_year1")
cutoff

In [ ]:
imp = year["import_kwh"].groupby(year.index.hour).sum()
imp.plot(kind="bar", figsize=(9, 3), title="Annual grid import by hour (kWh)")

In [ ]:
def shift_import(usage, kwh_per_day, src_hours, dst_hours):
    """Move up to kwh_per_day of import from src_hours into dst_hours each day.
    src_hours must not overlap dst_hours. Keep kwh_per_day <= charge_kW x len(dst_hours)."""
    u = usage.copy()
    for _, g in u.groupby(u.index.date):
        src = g[g.index.hour.isin(src_hours)]
        total = src["import_kwh"].sum()
        move = min(kwh_per_day, total)
        if move <= 0:
            continue
        u.loc[src.index, "import_kwh"] -= src["import_kwh"] * move / total
        dst = g.index[g.index.hour.isin(dst_hours)]
        u.loc[dst, "import_kwh"] += move / len(dst)
    return u

nb_plan = plans["electric_ireland_night_boost"]
for kwh in [0, 2, 4, 5]:
    shifted = shift_import(year, kwh, src_hours=[23, 0, 1, 4, 5, 6, 7], dst_hours=[2, 3])
    print(kwh, round(bill_summary(nb_plan, shifted, days=365, charges=charges)["net_cost"]))

In [ ]:
EVENING = [19, 20, 21, 22]
night_all = [23, 0, 1, 2, 3, 4, 5, 6, 7]
dst = {
    "electric_ireland_current": night_all,
    "electric_ireland_sst_saver_16": night_all,
    "electric_ireland_night_boost": [2, 3],
    "sse_airtricity_smart_ev_max": [23, 0, 1, 2, 3, 4],
    "sse_airtricity_top_discount_smart": night_all,
    "energia_smart_data": night_all,
    "energia_ev_smart_drive_plus": [2, 3, 4, 5],
    "bord_gais_smart_ev_plus": [2, 3, 4],
}
out = {}
for name, hrs in dst.items():
    p = stay if name == "electric_ireland_current" else plans[name]
    out[name] = {k: bill_summary(p, shift_import(year, k, EVENING, hrs), days=365, charges=charges)["net_cost"]
                 for k in [0, 1, 2, 3]}
pd.DataFrame(out).T.round(0)